In [44]:
##根據面積變化 重新判斷G S 並顯示比例

import arcpy
from collections import Counter

# ==========================================
# 1. 參數設定
# ==========================================
t1_name = "OL_P1_2204_coral_Simplify"     # 含有 Fate 欄位的圖層 (Target)
t2_name = "OL_P1_2209_cor_SimplifyPolyg"  # 用來比對面積的圖層
id_field = "Temp_ID"
fate_field = "Fate"

# 定義「需要被重新判斷」的類別名稱 (請確保與 Domain 中的描述完全一致)
TARGET_LABELS = ["G", "S", "GorS"]

# ==========================================
# 2. 自動獲取 Domain 映射
# ==========================================
def get_domain_info(layer, field_name):
    desc = arcpy.Describe(layer)
    workspace = desc.catalogPath.split('.gdb')[0] + '.gdb'
    f_obj = [f for f in arcpy.ListFields(layer) if f.name == field_name][0]
    
    if not f_obj.domain:
        print("⚠️ 警告：未偵測到 Domain，將以純字串模式執行。")
        return {"G": "G", "S": "S", "GorS": "GorS"}, {"G": "G", "S": "S", "GorS": "GorS"}

    domains = arcpy.da.ListDomains(workspace)
    coded_dict = next(d.codedValues for d in domains if d.name == f_obj.domain)
    inv_map = {v: k for k, v in coded_dict.items()} # {描述: 代碼}
    return inv_map, coded_dict

val_to_code, code_to_label = get_domain_info(t1_name, fate_field)

# 準備更新用的代碼
CODE_G = val_to_code.get("G")
CODE_S = val_to_code.get("S")
# 將需要判斷的標籤轉為代碼清單，以便後續過濾
TARGET_CODES = [val_to_code.get(label) for label in TARGET_LABELS if label in val_to_code]

# ==========================================
# 3. 讀取數據與分析
# ==========================================
t2_areas = {r[0]: r[1] for r in arcpy.da.SearchCursor(t2_name, [id_field, "SHAPE@AREA"]) if r[0]}

updates_plan = {}
before_list = []
after_preview_list = []

with arcpy.da.SearchCursor(t1_name, ["OID@", id_field, "SHAPE@AREA", fate_field]) as cursor:
    for oid, tid, t1_area, old_code in cursor:
        before_list.append(old_code)
        
        # 邏輯判斷：
        # 1. T2 必須有對應 ID
        # 2. 原始 Fate 必須在 G, S, GorS 的名單內 (避免改到『死亡』等其他類別)
        if tid in t2_areas and old_code in TARGET_CODES:
            t2_area = t2_areas[tid]
            new_code = CODE_G if (t2_area - t1_area > 0) else CODE_S
            
            after_preview_list.append(new_code)
            if new_code != old_code:
                updates_plan[oid] = new_code
        else:
            # 不符合更新條件的，維持原樣
            after_preview_list.append(old_code)

# ==========================================
# 4. 顯示全體 Fate 報表 (包含所有種類)
# ==========================================
def display_full_report(before_data, after_data, labels_dict):
    total = len(before_data)
    b_counts = Counter(before_data)
    a_counts = Counter(after_data)
    
    # 取得數據中出現過的所有代碼，並排序
    all_present_codes = sorted(list(set(list(b_counts.keys()) + list(a_counts.keys()))))
    
    print("\n" + "="*65)
    print(f"{'Fate 種類':<15} | {'更改前 (Before)':<20} | {'預計更改後 (After)':<20}")
    print("-" * 65)
    
    for code in all_present_codes:
        label = labels_dict.get(code, f"Unknown({code})")
        
        b_cnt = b_counts.get(code, 0)
        a_cnt = a_counts.get(code, 0)
        
        b_pct = (b_cnt / total * 100) if total > 0 else 0
        a_pct = (a_cnt / total * 100) if total > 0 else 0
        
        # 標註哪些類別發生了變動
        change_tag = " <--" if b_cnt != a_cnt else ""
        
        b_str = f"{b_cnt:>6} ({b_pct:>5.1f}%)"
        a_str = f"{a_cnt:>6} ({a_pct:>5.1f}%)"
        print(f"{str(label):<17} | {b_str:<20} | {a_str:<20}{change_tag}")
    
    print("-" * 65)
    print(f"總要素數: {total} | 預計修改筆數: {len(updates_plan)}")
    print("註：僅針對 G, S, GorS 進行面積邏輯判斷，其餘類別保持不變。")
    print("="*65)

display_full_report(before_list, after_preview_list, code_to_label)

# ==========================================
# 5. 確認與執行
# ==========================================
if len(updates_plan) > 0:
    if input("\n[詢問] 是否確認執行 T1 圖層的 Fate 更新？(y/n): ").lower() == 'y':
        with arcpy.da.UpdateCursor(t1_name, ["OID@", fate_field]) as u_cursor:
            for row in u_cursor:
                if row[0] in updates_plan:
                    row[1] = updates_plan[row[0]]
                    u_cursor.updateRow(row)
        print("\n>>> 更新成功！")
    else:
        print("\n>>> 操作已取消。")
else:
    print("\n>>> 無須更改任何資料。")


Fate 種類         | 更改前 (Before)         | 預計更改後 (After)       
-----------------------------------------------------------------
D                 |    487 ( 31.3%)      |    487 ( 31.3%)     
G                 |    467 ( 30.0%)      |    467 ( 30.0%)     
S                 |    260 ( 16.7%)      |    261 ( 16.8%)      <--
Fi                |     86 (  5.5%)      |     86 (  5.5%)     
GorS              |      1 (  0.1%)      |      0 (  0.0%)      <--
Fu                |    256 ( 16.4%)      |    256 ( 16.4%)     
-----------------------------------------------------------------
總要素數: 1557 | 預計修改筆數: 1
註：僅針對 G, S, GorS 進行面積邏輯判斷，其餘類別保持不變。



[詢問] 是否確認執行 T1 圖層的 Fate 更新？(y/n):  y



>>> 更新成功！


In [47]:
##計算各命運T2的面積與周長

import arcpy

# ==========================================
# 1. 參數設定
# ==========================================
t1_name = "OL_P1_2204_coral_Simplify"
t2_name = "OL_P1_2209_cor_SimplifyPolyg"

id_field = "Temp_ID"
fate_field = "Fate"
ramet_field = "Ramet" # 格式如 "01-02-03"

# 新增欄位名稱
area_t2_f = "Shape_Area_T2"
len_t2_f = "Shape_Length_T2"

# ==========================================
# 2. 準備工作：新增欄位與獲取 Domain
# ==========================================
# 新增欄位 (若已存在會跳過)
existing_fields = [f.name for f in arcpy.ListFields(t1_name)]
for f_name in [area_t2_f, len_t2_f]:
    if f_name not in existing_fields:
        arcpy.management.AddField(t1_name, f_name, "DOUBLE")
        print(f"新增欄位: {f_name}")

# 獲取 Domain 對照表 (沿用之前的邏輯)
def get_domain_map(layer, field_name):
    desc = arcpy.Describe(layer)
    workspace = desc.catalogPath.split('.gdb')[0] + '.gdb'
    f_obj = [f for f in arcpy.ListFields(layer) if f.name == field_name][0]
    if not f_obj.domain: return {}, {}
    domains = arcpy.da.ListDomains(workspace)
    coded_dict = next(d.codedValues for d in domains if d.name == f_obj.domain)
    return {v: k for k, v in coded_dict.items()}, coded_dict

val_to_code, _ = get_domain_map(t1_name, fate_field)

# 定義需要比對的代碼
C_D = val_to_code.get("D")
C_G = val_to_code.get("G")
C_S = val_to_code.get("S")
C_Fi = val_to_code.get("Fi")
C_Fu = val_to_code.get("Fu")

# ==========================================
# 3. 建立 T2 索引字典 {ID: (Area, Length)}
# ==========================================
# 使用 SHAPE@ 令牌獲取精確幾何數值
t2_lookup = {}
with arcpy.da.SearchCursor(t2_name, [id_field, "SHAPE@AREA", "SHAPE@LENGTH"]) as cursor:
    for tid, area, length in cursor:
        if tid:
            t2_lookup[str(tid)] = (area, length)

print(f"-> T2 索引建立完成，共 {len(t2_lookup)} 筆資料。")

# ==========================================
# 4. 執行 T1 更新
# ==========================================
exceptions = [] # 儲存錯誤訊息
success_count = 0

# 需要處理的欄位
fields = ["OID@", fate_field, id_field, ramet_field, area_t2_f, len_t2_f]

with arcpy.da.UpdateCursor(t1_name, fields) as cursor:
    for row in cursor:
        oid, fate, tid, ramet, _, _ = row
        tid_str = str(tid) if tid else ""
        
        target_area = 0.0
        target_len = 0.0
        found_data = True
        
        # --- 邏輯判斷開始 ---
        
        # Case 1: 死亡 (D)
        if fate == C_D:
            target_area = 0.0
            target_len = 0.0
            
        # Case 2: 生長(G), 縮小(S), 融合(Fu) -> 直接找 T2 對應 ID
        elif fate in [C_G, C_S, C_Fu]:
            if tid_str in t2_lookup:
                target_area, target_len = t2_lookup[tid_str]
            else:
                found_data = False
                exceptions.append(f"OID {oid}: 找不到 Temp_ID '{tid_str}' (Fate: G/S/Fu)")
                
        # Case 3: 分裂 (Fi) -> 拆解 Ramet 欄位並累加
        elif fate == C_Fi:
            if ramet:
                parts = ramet.split("-")
                sub_area = 0.0
                sub_len = 0.0
                for p in parts:
                    p = p.strip()
                    if p in t2_lookup:
                        sub_area += t2_lookup[p][0]
                        sub_len += t2_lookup[p][1]
                    else:
                        found_data = False
                        exceptions.append(f"OID {oid}: Ramet 中的 ID '{p}' 在 T2 中不存在")
                target_area, target_len = sub_area, sub_len
            else:
                found_data = False
                exceptions.append(f"OID {oid}: Fate 為 Fi 但 Ramet 欄位為空")
        
        # 其餘不處理的情況 (保持原本數值或設為 None)
        else:
            continue 

        # --- 執行寫入 ---
        if found_data:
            row[4] = target_area
            row[5] = target_len
            cursor.updateRow(row)
            success_count += 1

# ==========================================
# 5. 回報結果
# ==========================================
print("\n" + "="*50)
print(f"處理完成！")
print(f"成功更新筆數: {success_count}")
print(f"例外錯誤筆數: {len(exceptions)}")
print("="*50)

if exceptions:
    print("\n[例外狀況清單]")
    # 只列出前 10 筆，避免洗版
    for ex in exceptions[:10]:
        print(f"  - {ex}")
    if len(exceptions) > 10:
        print(f"  ... 以及其他 {len(exceptions)-10} 筆錯誤")

-> T2 索引建立完成，共 1223 筆資料。

處理完成！
成功更新筆數: 1557
例外錯誤筆數: 0


In [48]:
## 計算面積和型態變化率

import arcpy
import math

# ==========================================
# 1. 參數與環境設定
# ==========================================
t1_name = "OL_P1_2204_coral_Simplify"
t1_id_field = "ID"            # Fu 查詢 T1 使用的 Long 欄位
fate_field = "Fate"
fu_group_field = "Fu_group"   # 格式範例: 01+02+03

# 定義基礎面積與周長欄位
a1_f, l1_f = "Shape_Area", "Shape_Length"
a2_f, l2_f = "Shape_Area_T2", "Shape_Length_T2" # 需確保此欄位已存在並填值

# ==========================================
# 2. 欄位初始化
# ==========================================
calc_fields = ["Growth_Rate_Yan", "Circ_T1", "Circ_T2", "Circ_Rate_Yan", "PAratio_Rate_Yan"]
existing = [f.name for f in arcpy.ListFields(t1_name)]
for f in calc_fields:
    if f not in existing:
        arcpy.management.AddField(t1_name, f, "DOUBLE")

# 建立 T1 幾何索引 (用於 Fu 聚合計算 T1 的虛擬初始狀態)
# 結構: {ID: (面積, 周長)}
t1_geom_index = {int(r[0]): (r[1], r[2]) for r in arcpy.da.SearchCursor(t1_name, [t1_id_field, a1_f, l1_f]) if r[0] is not None}

# 獲取 Domain 代碼 (D, S, G, Fi, Fu)
def get_domain_map(layer, field_name):
    desc = arcpy.Describe(layer)
    workspace = desc.catalogPath.split('.gdb')[0] + '.gdb'
    f_obj = [f for f in arcpy.ListFields(layer) if f.name == field_name][0]
    coded_dict = next(d.codedValues for d in arcpy.da.ListDomains(workspace) if d.name == f_obj.domain)
    return {v: k for k, v in coded_dict.items()}

val_to_code = get_domain_map(t1_name, fate_field)
C_D, C_S, C_G, C_Fi, C_Fu = [val_to_code.get(x) for x in ["D", "S", "G", "Fi", "Fu"]]

# ==========================================
# 3. 核心運算邏輯
# ==========================================
success_count = 0
exceptions = []

# 準備 UpdateCursor 欄位清單
update_fields = ["OID@", fate_field, a1_f, l1_f, a2_f, l2_f, fu_group_field, 
                 "Growth_Rate_Yan", "Circ_T1", "Circ_T2", "Circ_Rate_Yan", "PAratio_Rate_Yan"]

with arcpy.da.UpdateCursor(t1_name, update_fields) as cursor:
    for row in cursor:
        oid, fate, a1, l1, a2, l2, fu_text, _, _, _, _, _ = row
        
        # 基本分母防錯
        if not a1 or a1 == 0: continue
            
        # --- [步驟 A] 計算單體的基礎指標 ---
        # 圓度公式: 4πA / P^2 | PA Ratio公式: P / A
        c1 = (4 * math.pi * a1) / (l1**2) if l1 and l1 > 0 else 0
        pa1 = l1 / a1
        
        # T2指標 (Fi的a2/l2已是聚合值；Fu則是融合後的單體)
        c2 = (4 * math.pi * a2) / (l2**2) if l2 and l2 > 0 else 0
        pa2 = l2 / a2 if a2 and a2 > 0 else 0
        
        row[8], row[9] = c1, c2 # 更新 Circ_T1, Circ_T2
        
        g_rate = c_rate = pa_rate = 0.0
        is_calc = False

        try:
            # --- [步驟 B] 根據 Fate 計算變化率 ---
            
            # Case 1: D, S, G, Fi (Fi 的 T2 資料已在前期加總，可視為整體)
            if fate in [C_D, C_S, C_G, C_Fi]:
                g_rate = a2 / a1
                c_rate = c2 / c1 if c1 > 0 else 0
                pa_rate = pa2 / pa1 if pa1 > 0 else 0
                is_calc = True

            # Case 2: Fu (融合 - 核心修正邏輯)
            elif fate == C_Fu:
                if fu_text:
                    # 拆解融合群組 ID
                    ids = [int(i.strip()) for i in str(fu_text).split("+") if i.strip().isdigit()]
                    sum_a1 = sum_l1 = 0.0
                    all_found = True
                    
                    # 聚合計算 T1 所有成員的總面積與總周長
                    for tid in ids:
                        if tid in t1_geom_index:
                            sum_a1 += t1_geom_index[tid][0]
                            sum_l1 += t1_geom_index[tid][1]
                        else:
                            all_found = False
                    
                    if all_found and sum_a1 > 0:
                        # 計算 T1 群組的虛擬聚合圓度與 PA Ratio
                        c1_agg = (4 * math.pi * sum_a1) / (sum_l1**2) if sum_l1 > 0 else 0
                        pa1_agg = sum_l1 / sum_a1
                        
                        # 變化率 = 單一 T2 / 聚合 T1
                        g_rate = a2 / sum_a1
                        c_rate = c2 / c1_agg if c1_agg > 0 else 0
                        pa_rate = pa2 / pa1_agg if pa1_agg > 0 else 0
                        is_calc = True
                else:
                    exceptions.append(f"OID {oid}: Fu_group 欄位缺失")

            # 寫入結果
            if is_calc:
                row[7], row[10], row[11] = g_rate, c_rate, pa_rate
                cursor.updateRow(row)
                success_count += 1

        except Exception as e:
            exceptions.append(f"OID {oid}: {str(e)}")

# ==========================================
# 4. 報表回報
# ==========================================
print(f"\n[運算完成] 成功更新: {success_count} 筆 | 例外: {len(exceptions)} 筆")
if exceptions:
    print("例外範例:", exceptions[:5])


[運算完成] 成功更新: 1557 筆 | 例外: 0 筆


In [49]:
## 加入時間標準化

import arcpy
import math

# ==========================================
# 1. 參數設定 (請直接在此修改變數)
# ==========================================
t1_name = "OL_P1_2204_coral_Simplify"

# 調查日期設定 (格式: YYYY/MM)
t1_date_str = "2022/04"
t2_date_str = "2022/09"

# 原始欄位對應新的年化欄位
field_map = {
    "Growth_Rate_Yan": "Growth_Rate_Year",
    "Circ_Rate_Yan": "Circ_Rate_Year",
    "PAratio_Rate_Yan": "PAratio_Year"
}

# ==========================================
# 2. 時間解析與年化係數計算
# ==========================================
# 解析日期字串
y1, m1 = map(int, t1_date_str.split('/'))
y2, m2 = map(int, t2_date_str.split('/'))

# 計算總月數 N
total_months = (y2 - y1) * 12 + (m2 - m1)

if total_months <= 0:
    raise ValueError(f"日期設定錯誤：T2 ({t2_date_str}) 必須晚於 T1 ({t1_date_str})")

# 計算年化指數 (12 / N)
annual_power = 12.0 / total_months

print(f"-> 調查期間：{t1_date_str} 至 {t2_date_str}")
print(f"-> 總月數間隔：{total_months} 個月")
print(f"-> 年化修正指數 (12/N)：{annual_power:.4f}")

# ==========================================
# 3. 欄位準備與更新
# ==========================================
# 新增年化欄位 (若不存在)
existing = [f.name for f in arcpy.ListFields(t1_name)]
for new_f in field_map.values():
    if new_f not in existing:
        arcpy.management.AddField(t1_name, new_f, "DOUBLE")

# 準備 UpdateCursor 欄位
src_fields = list(field_map.keys())
dst_fields = list(field_map.values())
all_fields = src_fields + dst_fields

success_count = 0
with arcpy.da.UpdateCursor(t1_name, all_fields) as cursor:
    for row in cursor:
        # row[0-2]: 原始總變化率, row[3-5]: 年化結果
        for i in range(3):
            total_rate = row[i]
            
            # 只有當總速率 > 0 時才進行幾何年化計算
            if total_rate is not None and total_rate > 0:
                # 公式: Rate_Annual = Rate_Total ^ (12 / N)
                row[i+3] = math.pow(total_rate, annual_power)
            elif total_rate == 0:
                row[i+3] = 0
            else:
                row[i+3] = None # 處理無效或負值數據
        
        cursor.updateRow(row)
        success_count += 1

# ==========================================
# 4. 結果回報
# ==========================================
print("-" * 50)
print(f"年平均變化率標準化完成！")
print(f"成功處理: {success_count} 筆資料")
print("-" * 50)

-> 調查期間：2022/04 至 2022/09
-> 總月數間隔：5 個月
-> 年化修正指數 (12/N)：2.4000
--------------------------------------------------
年平均變化率標準化完成！
成功處理: 1557 筆資料
--------------------------------------------------


In [ ]:
##加入簡化的G S判定